
# Lecture 2 — Numerical Optimization in DICE (Python)

Ce cahier complète **Lecture 2** sur la tarification du carbone et le contrôle optimal. Nous passons de l'élaboration de scénarios (*ce qui pourrait arriver*) au problème du planificateur social (*ce qui devrait arriver*), et nous montrons comment le prix optimal du carbone (CSC)** émerge et est calculé numériquement.

**Roadmap**  
1. Charger le modèle et inspecter la structure d'état/de contrôle.
2. Rétablir l'objectif** du planificateur et la troncation temporelle (approximation finite-horizon).
3. Résoudre numériquement avec une fenêtre de planification de roulement; **lire SCC** à partir des sorties.
4. Sensibilité (p. ex. dommages plus élevés) et résultats d'exportation.



## 1. Setup
Nous importons les composants du modèle DICE et nous veillons à ce que Python puisse trouver le module.


In [ ]:
# Run this cell once to check/install the Python packages required for this notebook.

import sys
import subprocess
import importlib.util

required = {
    "matplotlib": "matplotlib",
    "numba": "numba",
    "numpy": "numpy",
    "pandas": "pandas",
    "scipy": "scipy",
    "tqdm": "tqdm",
}

missing = [
    package
    for module, package in required.items()
    if importlib.util.find_spec(module) is None
]

if missing:
    print("Installing:", ", ".join(missing))
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", *missing]
    )

print("Python environment ready.")


In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from DICE import (
    Params,
    init_states,
    update_path,
    mat_to_df,
    obj_fun,
    run_optimal_policy,
)

p = Params()
print("DICE parameters loaded; periods:", p.nT)



## 2. Modèle squelette (états et contrôles)

- ** États (exemples):** capital \(K t\), TFP \(A t\), population \(L t\), stocks climatiques \(M {\mathrm{AT}}, M {\mathrm{UP}}, M {\mathrm{LO}}\), températures \(T {\mathrm{AT}}, T {\mathrm{LO}}\).
- **Contrôles:** taux d'épargne \(s t\), taux de réduction \(\mu t\) (qui, avec les paramètres technologiques, implique une trajectoire ** prix carbone / taxe**).
- **Unités:** \(C t\) en trillions USD, \(L t\) en millions → consommation par habitant en **milliers USD**: \(c t = 1000 \times C t/L t\).


In [ ]:

# Instantiate parameters and construct an initial path
p = Params()
sim = init_states(p)

# Baseline controls for initialization (not optimal): small abatement, constant saving
sim[:, p.i_mu] = 0.03
sim[:, p.i_s]  = 0.20

# Update path for all endogenous variables given the controls
timevec = range(1, p.nT)
sim = update_path(sim, timevec, p)

# Peek at available columns
print("Columns:", p.col)
print("Horizon length (periods):", p.nT, " — years:", sim[-1, p.i_time])



## 3. Objectif du planificateur (utilité de l'ERCR)

We maximize discounted social welfare
\[
W = \sum_{i=0}^{I^\star} \beta_\Delta^{\,i}\, L_{t+i\Delta}\, u\!\left( 1000\,\frac{C_{t+i\Delta}}{L_{t+i\Delta}} \right), 
\qquad \beta_\Delta=(1+\rho)^{-\Delta},
\]
avec l'utilitaire CRA \(u(c)=\frac{c^{1-\gamma}-1}{1-\gamma}\). Le cahier utilise`obj_fun` fournies dans le module.

**Note sur la troncation:** Nous remplaçons l'horizon infini par un \(I^\star\) fini choisi de sorte que le poids de la queue est inférieur à une tolérance.


In [ ]:

# Examine the objective on a constant-control path (sanity check)
# Here we 'flatten' a control (e.g., saving only) and evaluate welfare.
x_const = np.full(p.nT-1, 0.20)  # constant saving
W_val = obj_fun(x_const, sim, timevec, p, [p.i_s])
print("Objective value (negative welfare) on constant s=0.20 path:", W_val)



## 4. Finite-horizon planning window \(I^\star\)

Nous sélectionnons une fenêtre de planification pour que le poids de réduction du dernier terme soit inférieur à la tolérance numérique`p.toly`. Ceci implémente l'approximation horizontale finie discutée dans les diapositives.


In [ ]:

# Compute planning window similar to DICE.run_optimal_policy
disc, Tplanner = 1.0, 1
while disc > p.toly:
    Tplanner += 1
    disc *= (1.0/(1.0 + p.rho))**p.Delta

print("Chosen planning window (periods):", Tplanner, " (each period = Δ years =", p.Delta, ")")



## 5. Solve a first problem: **savings only** (no transition control)

Nous optimisons uniquement le taux d'épargne \(s_t\), dans les bornes définies par le fichier de paramètres, en maintenant \(\mu_t\) fixé. Il s'agit du scénario de référence « sans transition ».


In [ ]:

print("Solving: savings-only optimal path...")
bounds_s   = (p.s_lower, p.s_upper)
control_id = [p.i_s]
path_opt_s = run_optimal_policy(sim.copy(), timevec, p, bounds_s, control_id)

print("Done. Preview of s_t path (first 5):", path_opt_s[:5, p.i_s])



## 6. Résoudre le problème : **économies + réduction**

Maintenant nous optimisons la paire \(s t, \mu t)\). L'évolution de la taxe sur le carbone est alors implicite par le modèle et indiquée dans le`Tax` column.


In [ ]:

print("Solving: joint (s_t, μ_t) optimal path...")
bounds_smu   = [(p.s_lower, p.s_upper), (0.0, 1.0)]
control_id   = [p.i_s, p.i_mu]
path_opt_smu = run_optimal_policy(sim.copy(), timevec, p, bounds_smu, control_id)

print("Done. Preview of μ_t path (first 5):", path_opt_smu[:5, p.i_mu])



## 7. Visualiser les contrôles optimaux et le prix implicite du carbone

Ci-dessous, nous posons (un graphique par figure) l'économie optimale \(s t\), la réduction \(\mu t\), et la taxe implicite sur le carbone (USD/tCO\( 2\))`Tax`.


In [ ]:
# Plot saving rate
plt.figure(figsize=(6,3.8))
plt.plot(path_opt_smu[:, p.i_time], path_opt_smu[:, p.i_s], label="Optimal transition (s_t)")
plt.plot(path_opt_s[:,  p.i_time], path_opt_s[:,  p.i_s],  linestyle="--", label="No transition (s_t)")
plt.xlabel("Année"); plt.ylabel("Saving rate s_t"); plt.title("Optimal saving rate")
plt.legend(); plt.grid(True); plt.tight_layout()


In [ ]:
# Plot abatement rate
plt.figure(figsize=(6,3.8))
plt.plot(path_opt_smu[:, p.i_time], path_opt_smu[:, p.i_mu], label="Optimal transition (μ_t)")
plt.xlabel("Année"); plt.ylabel("Abatement rate μ_t"); plt.title("Optimal abatement rate")
plt.legend(); plt.grid(True); plt.tight_layout()


In [ ]:
# Plot implied carbon tax (USD per tCO2)
plt.figure(figsize=(6,3.8))
plt.plot(path_opt_smu[:, p.i_time], path_opt_smu[:, p.i_Tax], label="Implied carbon tax (USD/tCO₂)")
plt.xlabel("Année"); plt.ylabel("Carbon tax (USD/tCO₂)"); plt.title("Optimal carbon price / SCC (model-implied)")
plt.legend(); plt.grid(True); plt.tight_layout()



## 8. Lecture du CSC (prix du carbone) de la solution

- Dans cette mise en œuvre, la colonne`Tax` indique le modèle appliqué **prix optimal du carbone** (USD/tCO\( 2\)).
- Conceptuellement, le CCN est le prix d'ombre** des émissions dans le problème du planificateur. Dans la pratique numérique, on peut aussi la récupérer en :
  1. Lecture du multiplicateur **KKT** sur la contrainte d'émission (si le solveur expose des duels), ou
  2. Exécuter une petite expérience de perturbation**: ajouter \(\varepsilon\) tonnes aux émissions au moment \(t\), recalculer le bien-être, et convertir la perte de bien-être marginal en USD en utilisant l'utilité marginale.

Voici un squelette minimal pour (2). Nous la gardons commentée pour la vitesse.


In [ ]:

# -- Optional: perturbation-based SCC at a chosen date (slow; left as a template) --
# t_idx = 10         # pick a period index
# eps  = 1e-3        # add 0.001 GtC (adjust units as needed) to emissions at t_idx
# 
# base = path_opt_smu.copy()
# pert = path_opt_smu.copy()
# 
# # Inject a tiny increase in emissions at t_idx by lowering abatement a notch
# # (Here just as an illustration; in a full experiment you'd alter E_t directly or add to the carbon cycle)
# pert[t_idx, p.i_mu] = max(0.0, pert[t_idx, p.i_mu] - 1e-5)
# 
# base_W = obj_fun(np.hstack([base[1:, p.i_s], base[1:, p.i_mu]]), base, range(1, p.nT), p, [p.i_s, p.i_mu])
# pert_W = obj_fun(np.hstack([pert[1:, p.i_s], pert[1:, p.i_mu]]), pert, range(1, p.nT), p, [p.i_s, p.i_mu])
# dW = pert_W - base_W  # (negative welfare) difference
# print("ΔW from ε-perturbation:", dW)
# # Convert to USD/tCO2 using marginal utility at t_idx if needed.



## 9. Sensitivity: higher damages

Nous pouvons changer la fonction de dommages par`p.user_damage_fn` et relancez l'optimisation pour voir comment le \(\mu t\) optimal et le prix du carbone répondent.


In [ ]:

def higher_damage(T, p_local: Params):
    # Slightly steeper quadratic damages as an illustration
    return 0.0030 * (np.asarray(T) ** 2)

p_sens = Params()
sim_sens = init_states(p_sens)
sim_sens[:, p_sens.i_mu] = 0.03
sim_sens[:, p_sens.i_s]  = 0.20
sim_sens = update_path(sim_sens, range(1, p_sens.nT), p_sens)

p_sens.user_damage_fn = higher_damage
path_opt_smu_highDam = run_optimal_policy(sim_sens.copy(), range(1, p_sens.nT), p_sens, [(p_sens.s_lower,p_sens.s_upper),(0.0,1.0)], [p_sens.i_s, p_sens.i_mu])

# Plot comparison for μ_t
plt.figure(figsize=(6,3.8))
plt.plot(path_opt_smu[:, p.i_time], path_opt_smu[:, p.i_mu], linestyle="--", label="Baseline damages")
plt.plot(path_opt_smu_highDam[:, p.i_time], path_opt_smu_highDam[:, p.i_mu], label="Higher damages")
plt.xlabel("Année"); plt.ylabel("Abatement rate μ_t"); plt.title("Damage sensitivity: optimal abatement")
plt.legend(); plt.grid(True); plt.tight_layout()

# Plot comparison for implied carbon tax
plt.figure(figsize=(6,3.8))
plt.plot(path_opt_smu[:, p.i_time], path_opt_smu[:, p.i_Tax], linestyle="--", label="Baseline damages")
plt.plot(path_opt_smu_highDam[:, p.i_time], path_opt_smu_highDam[:, p.i_Tax], label="Higher damages")
plt.xlabel("Année"); plt.ylabel("Carbon tax (USD/tCO₂)"); plt.title("Damage sensitivity: optimal carbon price")
plt.legend(); plt.grid(True); plt.tight_layout()



## 10. Export results

Convertir des tableaux en un rangé`DataFrame` pour plus d'analyse ou de tracé ailleurs.


In [ ]:

df_baseline = pd.DataFrame({
    "year": path_opt_smu[:, p.i_time],
    "s":    path_opt_smu[:, p.i_s],
    "mu":   path_opt_smu[:, p.i_mu],
    "tax":  path_opt_smu[:, p.i_Tax],
    "E":    path_opt_smu[:, p.i_E],
    "T_AT": path_opt_smu[:, p.i_T_AT],
    "C":    path_opt_smu[:, p.i_C],
})
df_baseline.head()



### Wrap-up

- Le prix optimum du carbone**`Tax` fournit l'orientation normative du modèle.
- La fenêtre **finite-horizon** implémente la troncation discutée dans la conférence.
- La solution **rolling** produit des chemins à plein temps pour les états et les contrôles.
- Les expériences de sensibilité (p. ex. dommages) changent directement \(\mu t\) et la trajectoire fiscale implicite.
